# Get the data

In [56]:
from sklearn.datasets import fetch_california_housing

dataset = fetch_california_housing()
X = dataset.data
y = dataset.target

print(X.shape, y.shape)

(20640, 8) (20640,)


# Split the data

In [57]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=43)
print(f'Shape of train: {X_train.shape}, {y_train.shape}')
print(f'Shape of test: {X_test.shape}, {y_test.shape}')

Shape of train: (15480, 8), (15480,)
Shape of test: (5160, 8), (5160,)


# Scaler

In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Turn the data into tensors

In [59]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print(f'Shape of train: {X_train.shape}, {y_train.shape}')
print(f'Shape of test: {X_test.shape}, {y_test.shape}')


Shape of train: torch.Size([15480, 8]), torch.Size([15480, 1])
Shape of test: torch.Size([5160, 8]), torch.Size([5160, 1])


# Prepare the train and test sets

In [60]:
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

# Create the model 1

In [61]:
import torch.nn as nn


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.modelo = nn.Sequential(
        # 1 HIDDEN LAYER: 128 NEURONS
        nn.Linear(8, 128),
        nn.ReLU(),

        # 2 HIDDEN LAYER: 64 NEURONS
        nn.Linear(128, 64),
        nn.ReLU(),

        # 3 HIDDEN LAYER: 32 NEURONS
        nn.Linear(64, 32),
        nn.ReLU(),

        # OUTPUT LAYER: 1 NEURON
        nn.Linear(32, 1))

    def forward(self, x):
        return self.modelo(x)
    
model = Model()
print(model)

Model(
  (modelo): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


# Setup the Options for the model

In [62]:
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
epochs = 100

# Train the model

In [63]:
model.train()
for epoch in range(epochs):
    cumulative_loss = 0

    for batch_X, batch_Y in train_loader:
        # Clear the gradient for the last time
        optimizer.zero_grad()

        # Do the predict
        y_preds = model(batch_X)

        # Calculate the loss
        loss = criterion(y_preds, batch_Y)
        
        # bgackprop
        loss.backward()

        # Update the heights
        optimizer.step()

        # Loss  for the epoch
        cumulative_loss += loss.item()
        
    average_epoch_loss = cumulative_loss / len(train_loader)
    print(f'=' * 50)
    print(f'Epoch: {epoch + 1} | Loss Média: {average_epoch_loss:.4f}')

Epoch: 1 | Loss Média: 1.1420
Epoch: 2 | Loss Média: 0.4337
Epoch: 3 | Loss Média: 0.3878
Epoch: 4 | Loss Média: 0.3611
Epoch: 5 | Loss Média: 0.3439
Epoch: 6 | Loss Média: 0.3313
Epoch: 7 | Loss Média: 0.3259
Epoch: 8 | Loss Média: 0.3124
Epoch: 9 | Loss Média: 0.3071
Epoch: 10 | Loss Média: 0.3028
Epoch: 11 | Loss Média: 0.2993
Epoch: 12 | Loss Média: 0.2947
Epoch: 13 | Loss Média: 0.2958
Epoch: 14 | Loss Média: 0.2872
Epoch: 15 | Loss Média: 0.2883
Epoch: 16 | Loss Média: 0.2845
Epoch: 17 | Loss Média: 0.2820
Epoch: 18 | Loss Média: 0.2794
Epoch: 19 | Loss Média: 0.2760
Epoch: 20 | Loss Média: 0.2763
Epoch: 21 | Loss Média: 0.2746
Epoch: 22 | Loss Média: 0.2715
Epoch: 23 | Loss Média: 0.2697
Epoch: 24 | Loss Média: 0.2696
Epoch: 25 | Loss Média: 0.2678
Epoch: 26 | Loss Média: 0.2644
Epoch: 27 | Loss Média: 0.2674
Epoch: 28 | Loss Média: 0.2628
Epoch: 29 | Loss Média: 0.2616
Epoch: 30 | Loss Média: 0.2600
Epoch: 31 | Loss Média: 0.2581
Epoch: 32 | Loss Média: 0.2578
Epoch: 33 | Loss 

# Test set

In [64]:
from sklearn.metrics import mean_absolute_error

model.eval()
with torch.no_grad():
    previsoes = model(X_test)

    y_pred = previsoes.numpy()

    mae = mean_absolute_error(y_test, y_pred)
    print(f"MAE: {mae:.4f}")
    print(f"In practice, either the model gets it wrong close to: ${mae * 100000:.2f}")

MAE: 0.3322
In practice, either the model gets it wrong close to: $33216.64
